<a href="https://colab.research.google.com/github/joshuasamuel123/EnergyNation/blob/main/energy_nation_risk_engines_and_overlays_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
Patched MPI Risk Engine (v03 → v03a)
- Adds province–sector LR shrinkage knob for Bayes (to prevent swamping)
- Points Cox to expanded, audit-friendly coefficients (incl. Greenfield/FOAK, cost_percentile)
- Supports cleantech naming variants (cleantech_flag vs cleantech_Yes dummy)
- Exposes blend weight as a knob
"""

import os
import pandas as pd
import numpy as np

# =========================
# PATHS & CONSTANTS
# =========================
# Default to /content (Colab-style). Override with environment variables if needed.
BAYES_COEFF_PATH = os.getenv("BAYES_COEFF_PATH", "/content/bayes_lr_regenerated_coefficients_expanded.csv")
COX_COEFF_PATH   = os.getenv("COX_COEFF_PATH",   "/content/cox_refit_coefficients_timesplit_expanded.csv")
INPUT_XLSX       = os.getenv("INPUT_XLSX",       "/content/mpi_2024_input.xlsx")
OUT_CSV          = os.getenv("OUT_CSV",          "/content/mpi_2024_scored.csv")
OUT_XLSX         = os.getenv("OUT_XLSX",         "/content/mpi_2024_scored.xlsx")

# Survival baseline at 5 years (confirmed)
S0_T = float(os.getenv("S0_T", "0.545332"))

# ======== CONFIG KNOBS ========
# Bayes: control province–sector interaction strength
PROVSEC_ENABLED = (os.getenv("PROVSEC_ENABLED", "true").lower() == "true")
PROVSEC_ALPHA   = float(os.getenv("PROVSEC_ALPHA", "0.6"))  # 1.0 = no shrink; 0.6–0.8 recommended

# Blending: weight on Bayes
BLEND_BAYES_W   = float(os.getenv("BLEND_BAYES_W", "0.50"))
# ==============================


def norm_str(x):
    if pd.isna(x):
        return "Unknown"
    s = str(x).strip()
    return s if s else "Unknown"


def start_bin_from_year(y):
    try:
        y = int(y)
    except Exception:
        return "Unknown"
    return str(y) if 2018 <= y <= 2024 else "Unknown"


def _load_maps():
    bayes_table = pd.read_csv(BAYES_COEFF_PATH)
    cox_table   = pd.read_csv(COX_COEFF_PATH)
    bayes_lr_map = dict(zip(bayes_table["feature_name"], bayes_table["LR"]))

    # Normalize cox covariate column name
    if "covariate" not in cox_table.columns and "index" in cox_table.columns:
        cox_table = cox_table.rename(columns={"index": "covariate"})
    cox_coef_map = dict(zip(cox_table["covariate"], cox_table["coef"]))
    return bayes_lr_map, cox_coef_map


def _cost_quintile_for_cox(p):
    if pd.isna(p):
        return None
    v = float(p)
    return int(min(0.9999, max(0.0, v)) * 5)


def compute_p_bayes(row, bayes_lr_map):
    """Naive Bayes log-odds with optional α-shrinkage on province–sector interaction."""
    feats = ["PRIOR"]

    # cleantech
    cle = norm_str(row.get("cleantech"))
    feats.append(f"cleantech_{cle if cle in ['Yes','No'] else 'Unknown'}")

    # cost quintile (dataset-relative qcut assigned upstream)
    cq = row.get("_cost_quintile_bayes")
    feats.append(f"cost_quintile_{int(cq) if pd.notna(cq) else 'Unknown'}")

    # group
    grp = norm_str(row.get("group"))
    feats.append(f"group_{grp}" if f"group_{grp}" in bayes_lr_map else "group_Unknown")

    # province & sector
    prov = norm_str(row.get("province"))
    feats.append(f"province_{prov}" if f"province_{prov}" in bayes_lr_map else "province_Unknown")

    sec = norm_str(row.get("sector"))
    feats.append(f"sector_{sec}" if f"sector_{sec}" in bayes_lr_map else "sector_Unknown")

    # start year bin
    sb = start_bin_from_year(row.get("start_year"))
    feats.append(f"start_bin_{sb}" if f"start_bin_{sb}" in bayes_lr_map else "start_bin_Unknown")

    # province–sector interaction (with shrinkage)
    ps = f"prov_sec_{prov}_{sec}"
    if PROVSEC_ENABLED and ps in bayes_lr_map:
        feats.append(ps)

    # flags: greenfield / FOAK
    gf = row.get("greenfield_flag")
    if pd.notna(gf):
        try:
            feats.append(f"greenfield_flag_{int(gf)}")
        except Exception:
            pass
    foak = row.get("FOAK_flag")
    if pd.notna(foak):
        try:
            feats.append(f"FOAK_flag_{int(foak)}")
        except Exception:
            pass

    # sum log LRs with optional α on prov_sec_*
    log_odds = 0.0
    for f in feats:
        v = bayes_lr_map.get(f, 1.0)
        if f.startswith("prov_sec_"):
            if not PROVSEC_ENABLED:
                v = 1.0
            elif PROVSEC_ALPHA != 1.0:
                v = v ** PROVSEC_ALPHA
        log_odds += np.log(v)

    p = 1.0 / (1.0 + np.exp(-log_odds))
    return float(p)


def compute_risk_score(row, cox_coef_map):
    """
    Cox risk = exp(eta); eta = sum(beta_j * x_j)

    - Supports either 'cleantech_Yes' dummy (preferred) or legacy 'cleantech_flag'
    - Uses continuous 'cost_percentile' when present (from re-fit)
    - Includes 'greenfield_flag' and 'FOAK_flag' from re-fit
    - Province/Sector terms are taken literally (e.g., 'province_BC', 'sector_Mining')
    """
    eta = 0.0

    # cleantech: Yes dummy preferred; fallback to cleantech_flag
    cle = norm_str(row.get("cleantech"))
    cle_yes = 1 if cle == "Yes" else 0
    if "cleantech_Yes" in cox_coef_map:
        eta += cox_coef_map.get("cleantech_Yes", 0.0) * cle_yes
    else:
        eta += cox_coef_map.get("cleantech_flag", 0.0) * cle_yes

    # cost: prefer continuous cost_percentile
    cp = row.get("cost_percentile")
    if pd.notna(cp):
        try:
            eta += cox_coef_map.get("cost_percentile", 0.0) * float(cp)
        except Exception:
            pass
    else:
        # backward compatibility: discretize
        try:
            v = float(row.get("cost_percentile"))
            cq = int(min(0.9999, max(0.0, v)) * 5)
            eta += cox_coef_map.get("cost_quintile", 0.0) * cq
        except Exception:
            pass

    # province & sector
    prov = norm_str(row.get("province"))
    sec  = norm_str(row.get("sector"))
    eta += cox_coef_map.get(f"province_{prov}", 0.0)
    eta += cox_coef_map.get(f"sector_{sec}",   0.0)

    # Greenfield / FOAK flags
    gf = row.get("greenfield_flag")
    if pd.notna(gf):
        try:
            eta += cox_coef_map.get("greenfield_flag", 0.0) * int(gf)
        except Exception:
            pass

    foak = row.get("FOAK_flag")
    if pd.notna(foak):
        try:
            eta += cox_coef_map.get("FOAK_flag", 0.0) * int(foak)
        except Exception:
            pass

    return float(np.exp(eta))


def run():
    # Load inputs
    df = pd.read_excel(INPUT_XLSX)

    # Load coefficient maps
    bayes_lr_map, cox_coef_map = _load_maps()

    # === Bayes cost quintile via qcut over dataset ===
    ranks = df["project_cost"].astype(float).rank(method="first")
    df["_cost_quintile_bayes"] = pd.qcut(ranks, 5, labels=[0, 1, 2, 3, 4]).astype("Int64")

    # --- p_bayes ---
    df["p_bayes"] = df.apply(lambda r: compute_p_bayes(r, bayes_lr_map), axis=1)

    # --- Cox ---
    df["risk_score"] = df.apply(lambda r: compute_risk_score(r, cox_coef_map), axis=1)
    df["years_remaining"] = (5.0 - df["reporting_years"]).clip(lower=0.25)
    df["p_cox"] = 1 - (S0_T ** df["risk_score"])

    # --- Blend ---
    w = BLEND_BAYES_W
    df["blended_prob"] = w * df["p_bayes"] + (1.0 - w) * df["p_cox"]
    df["priority_index"] = df["blended_prob"] / df["years_remaining"]

    # Rescale within filtered dataset for urgency
    pi_min = df["priority_index"].min()
    pi_max = df["priority_index"].max()
    if pi_max > pi_min:
        df["urgency_scale_(0-1)"] = (df["priority_index"] - pi_min) / (pi_max - pi_min)
    else:
        df["urgency_scale_(0-1)"] = 0.0

    df["power_ranking"] = 0.60 * df["blended_prob"] + 0.40 * df["urgency_scale_(0-1)"]

    # SAVE
    df.to_csv(OUT_CSV, index=False)
    df.to_excel(OUT_XLSX, index=False)

    # Meta for audit
    print(f"[META] PROVSEC_ENABLED={PROVSEC_ENABLED}, PROVSEC_ALPHA={PROVSEC_ALPHA}")
    print(f"[META] COX_COEFF_PATH={COX_COEFF_PATH}")
    print(f"[META] BAYES_COEFF_PATH={BAYES_COEFF_PATH}")
    print(f"[META] BLEND_BAYES_W={BLEND_BAYES_W}")
    print("Wrote:", OUT_CSV)
    print("Wrote:", OUT_XLSX)
    print("Rows:", len(df))
    return df


if __name__ == "__main__":
    _ = run()


[META] PROVSEC_ENABLED=True, PROVSEC_ALPHA=0.6
[META] COX_COEFF_PATH=/content/cox_refit_coefficients_timesplit_expanded.csv
[META] BAYES_COEFF_PATH=/content/bayes_lr_regenerated_coefficients_expanded.csv
[META] BLEND_BAYES_W=0.5
Wrote: /content/mpi_2024_scored.csv
Wrote: /content/mpi_2024_scored.xlsx
Rows: 364


In [ ]:
#!/usr/bin/env python3
"""
EV (Expected Value) + Developer Engine — N‑year version (default: 5 years)

This script generalizes your previous 3‑year engine to an arbitrary horizon.
It allocates a chosen total probability (p_bayes / p_cox / blended_prob) across
a list of candidate FID years and computes EV by year, cumulative EV, discounted EV,
and developer metrics (PV_MOIC_k, k_star, IRR_k, k_star_adj) using an S‑curve
spend path to FID.

Key properties:
- EV horizon is configurable: --ev-horizon-years (default 5).
- Candidate years are BASE_YEAR+1 .. BASE_YEAR+horizon.
- Annual probabilities are allocated with a softmax that favors earlier years when
  urgency is high and years_remaining is small. If inputs are missing, allocation defaults
  to equal shares.
- Developer engine auto-detects annual_p_* columns to compute blended_prob, E_FID_year,
  PV sums, PV_MOIC_k, k_star, IRR_k, and k_star_adj.
- Recomputes annual_p_* and EV_* on every run (deterministic behavior from this script alone).
- No placeholders; fully runnable via CLI or by importing run_combined().

CLI example:
  python ev_engine_dev_combined_5y.py \
    --input mpi_2024_scored.csv \
    --output mpi_2024_ev_dev_combined_5y.csv \
    --p-source blended_prob \
    --value-mode capex \
    --dev-fee-rate 0.05 \
    --ev-discount 0.00 \
    --dev-discount 0.13 \
    --dev-cost-pct 0.03 \
    --k 1.0 \
    --scurve S \
    --scurve-steepness 6.0 \
    --ev-horizon-years 5

Outputs:
- Writes a CSV with original columns + annual_p_YYYY, EV_YYYY, EV_cum, EV_disc,
  and developer metrics (blended_prob, E_FID_year, cost_basis, DevCost,
  PV_OUT_per_DevCost, PV_IN_per_DevCost_k, PV_MOIC_k, k_star, IRR_k, k_star_adj).

Notes:
- Numeric misses are left as NaN; text misses as "".
- BASE_YEAR is 2024 by default; can be overridden via CLI.
"""

from __future__ import annotations
import argparse
import math
from typing import Dict, List

import numpy as np
import pandas as pd

# =============================
# Defaults & Globals
# =============================
BASE_YEAR_DEFAULT = 2024

# EV engine defaults
DEFAULT_P_SOURCE = "blended_prob"    # or: p_bayes, p_cox
DEFAULT_VALUE_MODE = "capex"         # or: dev_fee, count
DEFAULT_DEV_FEE_RATE = 0.05          # for value_mode == dev_fee
DEFAULT_EV_DISCOUNT = 0.0            # EV discount (often 0.0)
DEFAULT_ALPHA0 = 0.75                # allocation softmax hyperparam
DEFAULT_ALPHA1 = 2.0                 # allocation softmax hyperparam
DEFAULT_EV_HORIZON_YEARS = 5         # <-- 5-year EV allocation by default

# Developer engine defaults
DEFAULT_DEV_DISCOUNT = 0.13          # developer PV discount
DEFAULT_DEV_COST_PCT = 0.03          # DevCost = dev_cost_pct * cost_basis
DEFAULT_K = 1.0                      # reimbursement-only MOIC at FID
DEFAULT_SCURVE = "S"                 # S-curve to FID
DEFAULT_SCURVE_STEEPNESS = 6.0       # logistic steepness for S-curve


# =============================
# Utilities
# =============================
def _safe_float(x, fallback=np.nan):
    try:
        if x is None or (isinstance(x, float) and math.isnan(x)):
            return fallback
        return float(x)
    except Exception:
        return fallback


def _ensure_numeric(df: pd.DataFrame, cols: List[str]) -> None:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")


def _ev_years(base_year: int, horizon_years: int) -> List[int]:
    start = int(base_year) + 1
    return list(range(start, start + int(horizon_years)))


# =============================
# EV ENGINE (annual probs + EV by year)
# =============================
def _softmax_weights_n(t_years_remaining: float, urgency_01: float,
                       alpha0: float, alpha1: float, n: int) -> np.ndarray:
    """
    Generalized softmax allocator over n candidate years (0..n-1 index),
    favoring earlier years as urgency increases and years_remaining decreases.
    """
    t = max(0.25, float(t_years_remaining))
    u = float(urgency_01)
    if n <= 1:
        return np.array([1.0], dtype=float)

    idx = np.arange(n, dtype=float)
    denom = max(1.0, n - 1.0)
    # Score declines with later idx; rises with urgency; lower t boosts early years.
    scores = alpha0 * (u * (1.0 - (idx / denom)) + 1.0 / (t + 0.5 * idx))
    scores *= float(alpha1)
    # Stable softmax
    m = scores.max()
    e = np.exp(scores - m)
    w = e / e.sum()
    return w.astype(float)


def _allocate_annual_probs_n(row: pd.Series, p_total: float, years: List[int],
                             alpha0: float, alpha1: float) -> List[float]:
    n = len(years)
    if not (p_total and p_total > 0):
        return [0.0] * n

    yrs = _safe_float(row.get("years_remaining"))
    urg = _safe_float(row.get("urgency_scale_(0-1)"))
    if not (math.isnan(yrs) or math.isnan(urg)):
        w = _softmax_weights_n(yrs, urg, alpha0, alpha1, n)
    else:
        w = np.full(n, 1.0 / n, dtype=float)
    return [float(p_total * wi) for wi in w]


def _pick_p_total(row: pd.Series, p_source: str) -> float:
    if p_source == "p_bayes":
        return _safe_float(row.get("p_bayes"), 0.0)
    if p_source == "p_cox":
        return _safe_float(row.get("p_cox"), 0.0)
    return _safe_float(row.get("blended_prob"), 0.0)  # default


def _value_proxy_amount(row: pd.Series, value_mode: str, dev_fee_rate: float) -> float:
    capex = _safe_float(row.get("project_cost"), np.nan)
    if value_mode == "dev_fee":
        return capex * float(dev_fee_rate) if not math.isnan(capex) else np.nan
    if value_mode == "count":
        return 1.0
    # default: capex
    return capex


def compute_ev_fields(df: pd.DataFrame,
                      base_year: int,
                      p_source: str = DEFAULT_P_SOURCE,
                      value_mode: str = DEFAULT_VALUE_MODE,
                      dev_fee_rate: float = DEFAULT_DEV_FEE_RATE,
                      discount_rate: float = DEFAULT_EV_DISCOUNT,
                      alpha0: float = DEFAULT_ALPHA0,
                      alpha1: float = DEFAULT_ALPHA1,
                      horizon_years: int = DEFAULT_EV_HORIZON_YEARS) -> pd.DataFrame:
    df = df.copy()

    # Ensure numeric
    _ensure_numeric(df, [
        "project_cost", "years_remaining", "urgency_scale_(0-1)",
        "blended_prob", "p_bayes", "p_cox",
    ])

    # Compute value proxy explicitly
    df["value_mode"] = value_mode
    df["value_proxy_amount"] = df.apply(lambda r: _value_proxy_amount(r, value_mode, dev_fee_rate), axis=1)

    # Choose total probability and allocate over dynamic years
    df["p_source_used"] = p_source
    years = _ev_years(base_year, horizon_years)
    allocs = df.apply(
        lambda r: _allocate_annual_probs_n(
            r,
            _pick_p_total(r, p_source),
            years,
            alpha0, alpha1
        ),
        axis=1
    )

    # Materialize annual_p_YYYY columns
    for i, y in enumerate(years):
        df[f"annual_p_{y}"] = allocs.apply(lambda a: a[i])

    # EV per year = P(year) * value_proxy_amount
    for y in years:
        df[f"EV_{y}"] = df[f"annual_p_{y}"].astype(float) * df["value_proxy_amount"].astype(float)

    # Cumulative EV and discounted EV to base_year
    ev_cols = [f"EV_{y}" for y in years]
    df["EV_cum"] = df[ev_cols].sum(axis=1, min_count=1)

    if discount_rate and discount_rate != 0.0:
        def _pv_row(r: pd.Series) -> float:
            total = 0.0
            for y in years:
                ev_y = _safe_float(r.get(f"EV_{y}"), 0.0)
                total += ev_y / ((1.0 + discount_rate) ** (y - base_year))
            return total
        df["EV_disc"] = df.apply(_pv_row, axis=1)
    else:
        df["EV_disc"] = df["EV_cum"]

    return df


# =============================
# DEVELOPER ENGINE (PV sums, k*, PV_MOIC_k, IRR_k)
# =============================
def pv_factor_to_base(year: int, base_year: int, r: float) -> float:
    """Present-value factor relative to base_year.
    If year <= base_year: compound forward; if year > base_year: discount back.
    """
    if year == base_year:
        return 1.0
    if year < base_year:
        return (1.0 + r) ** (base_year - year)  # compounding forward
    return 1.0 / ((1.0 + r) ** (year - base_year))


def scurve_weights(years: List[int], shape: str = DEFAULT_SCURVE, steepness: float = DEFAULT_SCURVE_STEEPNESS) -> Dict[int, float]:
    """Normalized spend weights across the provided integer years up to FID.
    Shapes:
      - "S": logistic S-curve (default)
      - "even": uniform
      - "front": front-loaded
      - "back": back-loaded
    """
    ys = list(sorted(set(int(y) for y in years)))
    n = len(ys)
    if n <= 0:
        return {}
    if n == 1:
        return {ys[0]: 1.0}

    idx = np.arange(n, dtype=float)
    x = (idx - idx.mean()) / max(1.0, idx.std())

    if shape == "even":
        w = np.ones(n, dtype=float)
    elif shape == "front":
        w = np.exp(-0.8 * idx)           # more mass earlier
    elif shape == "back":
        w = np.exp(0.8 * idx)            # more mass later
    else:  # "S"
        s = float(steepness)
        w = 1.0 / (1.0 + np.exp(-s * x))  # 0..1 S-curve level
        w = np.diff(np.concatenate([[0.0], w]))  # convert to per-year increments
        w = np.clip(w, 1e-12, None)       # guard numerical

    w = w / w.sum()
    return {int(y): float(wi) for y, wi in zip(ys, w)}


def _robust_irr(cashflows: Dict[int, float], base_year: int, guess: float = 0.1,
                lo: float = -0.95, hi: float = 1.5, tol: float = 1e-6, iters: int = 200) -> float:
    """Bisection IRR on expected cashflows (practical and numerically stable)."""
    def npv_at(rate: float) -> float:
        return sum(val / ((1.0 + rate) ** (t - base_year)) for t, val in cashflows.items())

    npv_lo = npv_at(lo)
    npv_hi = npv_at(hi)
    if math.isnan(npv_lo) or math.isnan(npv_hi):
        return np.nan

    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        v = npv_at(mid)
        if abs(v) < tol:
            return mid
        if npv_lo * v <= 0:
            hi, npv_hi = mid, v
        else:
            lo, npv_lo = mid, v
    return 0.5 * (lo + hi)


def _pick_cost_basis(row: pd.Series) -> float:
    """Prefer project_cost → imputed_cost → value_proxy_amount."""
    for c in ("project_cost", "imputed_cost", "value_proxy_amount"):
        v = _safe_float(row.get(c), np.nan)
        if not math.isnan(v):
            return v
    return np.nan


def compute_dev_fields(df_ev: pd.DataFrame,
                       base_year: int,
                       discount_rate: float = DEFAULT_DEV_DISCOUNT,
                       dev_cost_pct: float = DEFAULT_DEV_COST_PCT,
                       k: float = DEFAULT_K,
                       scurve_shape: str = DEFAULT_SCURVE,
                       scurve_steepness: float = DEFAULT_SCURVE_STEEPNESS) -> pd.DataFrame:
    df = df_ev.copy()

    # Detect candidate FID years from annual_p_* columns
    ev_years = sorted(int(c.split("_")[-1]) for c in df.columns if c.startswith("annual_p_"))

    _ensure_numeric(df, [*(f"annual_p_{y}" for y in ev_years),
                         "start_year","end_year",
                         "project_cost","imputed_cost","value_proxy_amount"])

    rows = []
    for _, row in df.iterrows():
        # Build probability mass by candidate year
        p_by_y = {y: _safe_float(row.get(f"annual_p_{y}"), 0.0) for y in ev_years}
        p_by_y = {y: p for y, p in p_by_y.items() if p > 0}
        blended_prob = sum(p_by_y.values())

        start_year = int(_safe_float(row.get("start_year"), base_year))
        cost_basis = _pick_cost_basis(row)
        dev_cost = (dev_cost_pct * cost_basis) if not math.isnan(cost_basis) else np.nan

        e_fid = (sum(y * p for y, p in p_by_y.items()) / blended_prob) if blended_prob > 0 else np.nan

        # Unitized PV sums and per-year cashflow shares
        sum_pv_out_unit, sum_pv_in_unit = 0.0, 0.0
        per_year_out_share, per_year_in_share = {}, {}

        for y, p_y in p_by_y.items():
            # Spend path from start_year..y inclusive
            years = list(range(start_year, int(y) + 1))
            wts = scurve_weights(years, shape=scurve_shape, steepness=scurve_steepness)

            # PV of outflows (unitized) for scenario y
            pv_out_unit_y = 0.0
            for t, w in wts.items():
                pv = w * pv_factor_to_base(t, base_year=base_year, r=discount_rate)
                pv_out_unit_y += pv
                per_year_out_share[t] = per_year_out_share.get(t, 0.0) + p_y * w
            sum_pv_out_unit += p_y * pv_out_unit_y

            # PV of inflow (unitized) at FID year y
            pv_in_unit_y = pv_factor_to_base(int(y), base_year=base_year, r=discount_rate)
            sum_pv_in_unit += p_y * pv_in_unit_y
            per_year_in_share[int(y)] = per_year_in_share.get(int(y), 0.0) + p_y

        # Break-even multiple and PV_MOIC at k
        k_star = np.nan
        pv_moic_k = np.nan
        if sum_pv_in_unit > 0 and sum_pv_out_unit > 0:
            k_star = sum_pv_out_unit / sum_pv_in_unit
            pv_moic_k = (k * sum_pv_in_unit) / sum_pv_out_unit

        # Build expected cashflows for IRR (scale by DevCost)
        cashflows: Dict[int, float] = {}
        keys = per_year_out_share.keys() | per_year_in_share.keys()
        min_y = min(keys, default=base_year)
        max_y = max(keys, default=base_year)
        for t in range(min_y, max_y + 1):
            out_share = per_year_out_share.get(t, 0.0)
            in_share  = per_year_in_share.get(t, 0.0)
            cf_t = 0.0
            if not math.isnan(dev_cost):
                cf_t += -dev_cost * out_share
                cf_t +=  dev_cost * k * in_share
            cashflows[t] = cf_t

        irr_k = _robust_irr(cashflows, base_year=base_year, guess=0.1) if any(abs(v) > 0 for v in cashflows.values()) else np.nan

        rows.append({
            "blended_prob": blended_prob,
            "E_FID_year": e_fid,
            "cost_basis": cost_basis,
            "DevCost": dev_cost,
            "PV_OUT_per_DevCost": sum_pv_out_unit,
            "PV_IN_per_DevCost_k": sum_pv_in_unit,
            "PV_MOIC_k": pv_moic_k,
            "k_star": k_star,
            "IRR_k": irr_k,
        })

    met = pd.DataFrame(rows)
    with np.errstate(divide='ignore', invalid='ignore'):
        met["k_star_adj"] = met["k_star"] / met["blended_prob"]
        met.loc[(met["blended_prob"].fillna(0.0) == 0.0), "k_star_adj"] = np.nan

    return pd.concat([df.reset_index(drop=True), met.reset_index(drop=True)], axis=1)


# =============================
# Orchestration
# =============================
def run_combined(
    input_csv: str,
    output_csv: str,
    base_year: int = BASE_YEAR_DEFAULT,
    p_source: str = DEFAULT_P_SOURCE,
    value_mode: str = DEFAULT_VALUE_MODE,
    dev_fee_rate: float = DEFAULT_DEV_FEE_RATE,
    ev_discount_rate: float = DEFAULT_EV_DISCOUNT,
    alpha0: float = DEFAULT_ALPHA0,
    alpha1: float = DEFAULT_ALPHA1,
    ev_horizon_years: int = DEFAULT_EV_HORIZON_YEARS,
    dev_discount_rate: float = DEFAULT_DEV_DISCOUNT,
    dev_cost_pct: float = DEFAULT_DEV_COST_PCT,
    k: float = DEFAULT_K,
    scurve_shape: str = DEFAULT_SCURVE,
    scurve_steepness: float = DEFAULT_SCURVE_STEEPNESS,
) -> str:
    # Read input
    df_in = pd.read_csv(input_csv)

    # Compute EV fields (recomputed every run for determinism)
    df_ev = compute_ev_fields(
        df_in,
        base_year=base_year,
        p_source=p_source,
        value_mode=value_mode,
        dev_fee_rate=dev_fee_rate,
        discount_rate=ev_discount_rate,
        alpha0=alpha0,
        alpha1=alpha1,
        horizon_years=ev_horizon_years,
    )

    # Compute developer fields
    df_out = compute_dev_fields(
        df_ev,
        base_year=base_year,
        discount_rate=dev_discount_rate,
        dev_cost_pct=dev_cost_pct,
        k=k,
        scurve_shape=scurve_shape,
        scurve_steepness=scurve_steepness,
    )

    # Write output
    df_out.to_csv(output_csv, index=False)
    return output_csv


# =============================
# CLI
# =============================
# def _build_cli() -> argparse.ArgumentParser:
#     p = argparse.ArgumentParser(description="EV + Developer Engine (N-year horizon; default 5).")
#     p.add_argument("--input",  required=True, help="Input CSV (e.g., mpi_2024_scored.csv)")
#     p.add_argument("--output", required=True, help="Output CSV (e.g., mpi_2024_ev_dev_combined_5y.csv)")
#     p.add_argument("--base-year", type=int, default=BASE_YEAR_DEFAULT, help="Base year for PV (default 2024)")
#     p.add_argument("--p-source", choices=["blended_prob", "p_bayes", "p_cox"], default=DEFAULT_P_SOURCE,
#                    help="Total probability source to allocate across candidate years")
#     p.add_argument("--value-mode", choices=["capex", "dev_fee", "count"], default=DEFAULT_VALUE_MODE,
#                    help="Value proxy for EV calculation (capex=project_cost; dev_fee=capex*rate; count=1 per FID)")
#     p.add_argument("--dev-fee-rate", type=float, default=DEFAULT_DEV_FEE_RATE,
#                    help="Dev fee rate when --value-mode=dev_fee (default 0.05)")
#     p.add_argument("--ev-discount", type=float, default=DEFAULT_EV_DISCOUNT,
#                    help="Discount rate for EV PV (default 0.0)")
#     p.add_argument("--alpha0", type=float, default=DEFAULT_ALPHA0, help="Allocator hyperparam alpha0 (default 0.75)")
#     p.add_argument("--alpha1", type=float, default=DEFAULT_ALPHA1, help="Allocator hyperparam alpha1 (default 2.0)")
#     p.add_argument("--ev-horizon-years", type=int, default=DEFAULT_EV_HORIZON_YEARS,
#                    help="Number of candidate FID years to allocate probability/EV over (default 5)")
#     p.add_argument("--dev-discount", type=float, default=DEFAULT_DEV_DISCOUNT,
#                    help="Developer PV discount rate (default 0.13)")
#     p.add_argument("--dev-cost-pct", type=float, default=DEFAULT_DEV_COST_PCT,
#                    help="DevCost percentage of cost_basis (default 0.03)")
#     p.add_argument("--k", type=float, default=DEFAULT_K, help="Reimbursement multiple at FID for PV_MOIC_k (default 1.0)")
#     p.add_argument("--scurve", choices=["S", "even", "front", "back"], default=DEFAULT_SCURVE,
#                    help="Spend profile shape from start_year to FID year (default S)")
#     p.add_argument("--scurve-steepness", type=float, default=DEFAULT_SCURVE_STEEPNESS,
#                    help="Logistic steepness for S-curve (default 6.0)")
#     return p


# def main() -> None:
#     ap = _build_cli()
#     args = ap.parse_args()

#     run_combined(
#         input_csv=args.input,
#         output_csv=args.output,
#         base_year=args.base_year,
#         p_source=args.p_source,
#         value_mode=args.value_mode,
#         dev_fee_rate=args.dev_fee_rate,
#         ev_discount_rate=args.ev_discount,
#         alpha0=args.alpha0,
#         alpha1=args.alpha1,
#         ev_horizon_years=args.ev_horizon_years,
#         dev_discount_rate=args.dev_discount,
#         dev_cost_pct=args.dev_cost_pct,
#         k=args.k,
#         scurve_shape=args.scurve,
#         scurve_steepness=args.scurve_steepness,
#     )


# if __name__ == "__main__":
#     main()

# Call run_combined directly with specified input and output paths
run_combined(
    input_csv="/content/mpi_2024_scored.csv",
    output_csv="/content/mpi_2024_ev_dev_combined_5y.csv"
)

'/content/mpi_2024_ev_dev_combined_5y.csv'

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Unified Overlay Runner
----------------------
Applies Indigenous and MPO approval-cap overlays (ind_only, mpo_only, combined)
to a pre-construction MPI cohort and recomputes probabilities and expected value.

Key features:
- All knobs at the top (easy to tweak).
- Integrates EV_disc column if present.
- Standardizes years_in_pipeline from t0_orig as: 2024 - t0_orig + 1.
- Removes ownership flag handling per latest direction.
- MPO overlay = hard cap on federal approvals window.
- Indigenous overlay = time compression + hazard multiplier for flagged projects.
- Combined overlay runs overlays in a chosen sequence (default: MPO → Indigenous).
- Generates results CSV + grouped summary CSVs + a Markdown report.
- Robust to missing optional columns; emits clear warnings instead of failing.

Assumptions (kept explicit):
- Base probability column is PROB_COL (default 'blended'). Values in [0,1].
- Project cost column is COST_COL (default 'project_cost') in *millions of CAD*.
- Optional discounted cost column 'EV_disc' represents a discounted project cost
  (present value); when present we compute EV_disc = p * EV_disc.
- If explicit approvals duration is not provided, we infer from t0_orig:
  approvals_years ≈ years_in_pipeline = REPORT_YEAR - t0_orig + 1.
- Time compression impacts probabilities via a simple odds scaling:
  odds' = odds * (HAZARD_MULTIPLIER ** (-delta_years)),
  where negative delta_years (faster reviews) increases odds when multiplier > 1.
"""

import argparse
import os
import sys
import math
from typing import List, Optional, Dict
import pandas as pd
import numpy as np
from datetime import datetime

# =========================
# ======== KNOBS ==========
# =========================
# Paths
INPUT_CSV = "/content/mpi_2024_ev_dev_combined_5y.csv"  # <-- set to your input file
OUTPUT_DIR = "overlay_outputs"               # results and reports will be saved here

# Which scenario to run: 'ind_only' | 'mpo_only' | 'combined' | 'all'
SCENARIO = "all"

# Columns (override here if your headings differ)
ID_COL = "unique_id"
PROB_COL = "blended_prob"          # base probability (0..1)
COST_COL = "project_cost"     # in millions CAD
DISCOUNTED_COST_COL = "EV_disc"  # OPTIONAL: discounted project cost (PV). If present, we compute EV_disc = p * EV_disc
PROVINCE_COL = "province"
SECTOR_COL = "sector"
GROUP_COL = "group"
FOAK_COL = "FOAK"             # 1/0 or True/False
CLEANTECH_COL = "cleantech"   # 1/0 or True/False
COST_BAND_COL = "cost_band"   # categorical
T0_ORIG_COL = "t0_orig"       # a YEAR like 2019, 2020, etc. Used to compute years_in_pipeline
APPROVAL_YEARS_COL = "reporting_years"  # OPTIONAL explicit approvals duration; falls back to years_in_pipeline if missing

# Indigenous flag detection: the first present column among this list will be used.
INDIGENOUS_FLAG_CANDIDATES = [
    "indigenous_flag", "indigenous", "has_indigenous_partner", "ind"
]

# Year anchor for years_in_pipeline
REPORT_YEAR = 2024

# MPO overlay params
APPROVAL_CAP_YEARS = 2  # hard cap on approvals for the MPO overlay

# Indigenous overlay params
TIME_COMPRESSION_YEARS = -1.0  # negative means "faster by 1 year"
HAZARD_MULTIPLIER = 1.10       # >1 means faster reviews increase survival odds per year compressed

# Combined overlay sequence; valid tokens are 'mpo' and 'ind'
COMBINED_ORDER: List[str] = ["mpo", "ind"]

# Output formatting
FLOAT_DECIMALS = 6

# =========================
# ======== LOGIC ==========
# =========================

def warn(msg: str):
    print(f"[WARN] {msg}", file=sys.stderr)

def info(msg: str):
    print(f"[INFO] {msg}")

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def coerce_bool_or_int(series: pd.Series) -> pd.Series:
    """Coerce heterogenous 1/0/True/False/yes/no to clean {0,1} ints."""
    if series.dtype == bool:
        return series.astype(int)
    # Lowercase strings like 'true', 'yes', etc.
    def _coerce(v):
        if pd.isna(v):
            return 0
        if isinstance(v, (int, np.integer)):
            return int(v != 0)
        if isinstance(v, (float, np.floating)):
            return int(v != 0.0)
        s = str(v).strip().lower()
        if s in ("1", "true", "t", "y", "yes"):
            return 1
        if s in ("0", "false", "f", "n", "no"):
            return 0
        # Default to 0 but warn once
        return 0
    return series.map(_coerce).astype(int)

def clip_prob(p: pd.Series) -> pd.Series:
    return p.clip(lower=0.0, upper=1.0)

def odds_from_prob(p: np.ndarray) -> np.ndarray:
    # Add epsilon to avoid division by zero
    eps = 1e-12
    return p / (1.0 - p + eps)

def prob_from_odds(o: np.ndarray) -> np.ndarray:
    return o / (1.0 + o)

def apply_time_shift_probability(base_p: pd.Series,
                                 years_delta: pd.Series,
                                 hazard_multiplier: float) -> pd.Series:
    """
    Apply an odds-scaling to represent time compression/expansion effect.
    If years_delta is negative (faster), odds are multiplied by hazard_multiplier**(-years_delta) > 1.
    If years_delta is positive (slower), odds are reduced accordingly.
    """
    p = base_p.values.astype(float)
    y = years_delta.values.astype(float)
    odds = odds_from_prob(p)
    scale = np.power(hazard_multiplier, -y)  # note the negative sign
    new_odds = odds * scale
    new_p = prob_from_odds(new_odds)
    return pd.Series(new_p, index=base_p.index).pipe(clip_prob)

def ensure_column(df: pd.DataFrame, col: str, default=np.nan) -> pd.DataFrame:
    if col not in df.columns:
        df[col] = default
    return df

def years_in_pipeline_from_t0(df: pd.DataFrame, t0_col: str, report_year: int) -> pd.Series:
    """
    per latest instruction:
      years_in_pipeline = REPORT_YEAR - t0_orig + 1
    """
    if t0_col not in df.columns:
        warn(f"'{t0_col}' missing; cannot compute years_in_pipeline. Will default to NaN.")
        return pd.Series([np.nan]*len(df), index=df.index)

    years = (report_year - pd.to_numeric(df[t0_col], errors="coerce") + 1)
    return years

def infer_approvals_years(df: pd.DataFrame) -> pd.Series:
    """
    Prefer APPROVAL_YEARS_COL if present; otherwise fall back to years_in_pipeline.
    """
    if APPROVAL_YEARS_COL in df.columns:
        vals = pd.to_numeric(df[APPROVAL_YEARS_COL], errors="coerce")
        if vals.isna().all():
            warn(f"'{APPROVAL_YEARS_COL}' present but all NA; falling back to derived years_in_pipeline.")
            yip = years_in_pipeline_from_t0(df, T0_ORIG_COL, REPORT_YEAR)
            return yip
        return vals
    else:
        yip = years_in_pipeline_from_t0(df, T0_ORIG_COL, REPORT_YEAR)
        return yip

def detect_indigenous_flag(df: pd.DataFrame) -> Optional[str]:
    for cand in INDIGENOUS_FLAG_CANDIDATES:
        if cand in df.columns:
            return cand
    return None

def compute_ev(prob: pd.Series, cost_millions: pd.Series) -> pd.Series:
    """
    EV (in millions CAD) = probability * cost (in millions CAD).
    """
    return (prob.astype(float) * pd.to_numeric(cost_millions, errors="coerce").fillna(0.0))

def scenario_labels(s: str) -> Dict[str, str]:
    if s == "ind_only":
        return {"name": "ind_only", "pretty": "Indigenous Only"}
    if s == "mpo_only":
        return {"name": "mpo_only", "pretty": "MPO Cap Only"}
    if s == "combined":
        return {"name": "mpi_ind", "pretty": "Combined (MPO → Indigenous)"}
    return {"name": s, "pretty": s}

def run_mpo_overlay(df: pd.DataFrame) -> pd.Series:
    """
    Universal overlay: cap approvals at APPROVAL_CAP_YEARS, then convert time delta to probability shift.
    """
    approvals = infer_approvals_years(df)
    cap = APPROVAL_CAP_YEARS
    new_years = np.minimum(approvals.fillna(np.inf).values, cap)
    years_delta = pd.Series(new_years - approvals.values, index=df.index)  # <= 0
    # Apply odds scaling to base probability
    p0 = pd.to_numeric(df[PROB_COL], errors="coerce").fillna(0.0).clip(0, 1)
    p1 = apply_time_shift_probability(p0, years_delta, HAZARD_MULTIPLIER)
    return p1

def run_indigenous_overlay(df: pd.DataFrame) -> pd.Series:
    """
    Selective overlay: apply time compression + hazard multiplier only to Indigenous-flagged rows.
    Others get base probability unchanged.
    """
    p0 = pd.to_numeric(df[PROB_COL], errors="coerce").fillna(0.0).clip(0, 1)
    flag_col = detect_indigenous_flag(df)
    if flag_col is None:
        warn("No Indigenous flag column found; ind_only overlay will behave like no-op.")
        return p0

    flags = coerce_bool_or_int(df[flag_col])
    years_delta = pd.Series([0.0]*len(df), index=df.index)
    # Apply only to flagged rows
    years_delta.loc[flags == 1] = TIME_COMPRESSION_YEARS
    p1 = apply_time_shift_probability(p0, years_delta, HAZARD_MULTIPLIER)
    return p1

def run_combined_overlay(df: pd.DataFrame) -> pd.Series:
    """
    Apply overlays in sequence given by COMBINED_ORDER.
    """
    p = pd.to_numeric(df[PROB_COL], errors="coerce").fillna(0.0).clip(0, 1).copy()
    tmp_df = df.copy()
    tmp_df["_work_p"] = p.values
    for step in COMBINED_ORDER:
        if step.lower() == "mpo":
            # run MPO using current _work_p as base
            tmp_df[PROB_COL] = tmp_df["_work_p"]
            tmp_df["_work_p"] = run_mpo_overlay(tmp_df)
        elif step.lower() == "ind":
            tmp_df[PROB_COL] = tmp_df["_work_p"]
            tmp_df["_work_p"] = run_indigenous_overlay(tmp_df)
        else:
            warn(f"Unknown combined step '{step}' — skipping.")
    return tmp_df["_work_p"].clip(0, 1)

def attach_standard_fields(df: pd.DataFrame) -> pd.DataFrame:
    """
    - Derive years_in_pipeline from t0_orig.
    - Ensure grouping columns exist (fill NA for grouping).
    """
    df = df.copy()
    df["years_in_pipeline"] = years_in_pipeline_from_t0(df, T0_ORIG_COL, REPORT_YEAR)

    # Ensure group-by columns exist
    for col in [PROVINCE_COL, SECTOR_COL, GROUP_COL, FOAK_COL, CLEANTECH_COL, COST_BAND_COL]:
        if col not in df.columns:
            warn(f"Grouping column '{col}' missing; filling with NA.")
            df[col] = np.nan

    # Clean FOAK / cleantech to 0/1 if possible
    try:
        df[FOAK_COL] = coerce_bool_or_int(df[FOAK_COL])
    except Exception:
        pass
    try:
        df[CLEANTECH_COL] = coerce_bool_or_int(df[CLEANTECH_COL])
    except Exception:
        pass

    return df

def compute_overlay_outputs(df: pd.DataFrame, p_overlay: pd.Series, scenario_key: str) -> pd.DataFrame:
    """
    Build a result dataframe with overlay prob/EV deltas and crossover flags.
    """
    out = df.copy()
    out = ensure_column(out, ID_COL, default=np.arange(len(out)))
    out = ensure_column(out, COST_COL, default=0.0)
    out = ensure_column(out, PROB_COL, default=0.0)

    p0 = pd.to_numeric(out[PROB_COL], errors="coerce").fillna(0.0).clip(0, 1)
    c0 = pd.to_numeric(out[COST_COL], errors="coerce").fillna(0.0)

    # Base EVs
    out["EV_base"] = compute_ev(p0, c0)

    # Discounted EVs if EV_disc column present
    has_disc = (DISCOUNTED_COST_COL in out.columns)
    if has_disc:
        disc_cost = pd.to_numeric(out[DISCOUNTED_COST_COL], errors="coerce").fillna(0.0)
        out["EV_disc_base"] = compute_ev(p0, disc_cost)

    # Overlay
    out["blended_overlay"] = p_overlay.clip(0, 1)
    out["delta_blended"] = out["blended_overlay"] - p0

    out["EV_overlay"] = compute_ev(out["blended_overlay"], c0)
    out["delta_EV"] = out["EV_overlay"] - out["EV_base"]

    if has_disc:
        out["EV_disc_overlay"] = compute_ev(out["blended_overlay"], disc_cost)
        out["delta_EV_disc"] = out["EV_disc_overlay"] - out.get("EV_disc_base", 0.0)

    # Crossover cohorts
    out["cross_over_to_favorite"] = ((p0 < 0.5) & (out["blended_overlay"] >= 0.5)).astype(int)
    out["cross_out_to_underdog"] = ((p0 >= 0.5) & (out["blended_overlay"] < 0.5)).astype(int)

    # Label scenario
    meta = scenario_labels(scenario_key)
    out["scenario_key"] = meta["name"]
    out["scenario_label"] = meta["pretty"]

    # Tidy floats
    float_cols = out.select_dtypes(include=[float]).columns
    out[float_cols] = out[float_cols].astype(float).round(FLOAT_DECIMALS)

    return out

def group_summaries(df: pd.DataFrame, scenario_key: str) -> Dict[str, pd.DataFrame]:
    """
    Produce grouped summaries by province, sector, group, FOAK, cleantech, cost_band, plus overall.
    Summaries include counts, mean delta_blended, sums of delta_EV and delta_EV_disc, and crossover counts.
    """
    groups = {
        "province": [PROVINCE_COL],
        "sector": [SECTOR_COL],
        "group": [GROUP_COL],
        "foak": [FOAK_COL],
        "cleantech": [CLEANTECH_COL],
        "cost_band": [COST_BAND_COL],
        "overall": []  # special case
    }

    metrics = ["delta_blended", "delta_EV", "cross_over_to_favorite", "cross_out_to_underdog"]
    if "delta_EV_disc" in df.columns:
        metrics.append("delta_EV_disc")

    summaries = {}

    for name, cols in groups.items():
        if len(cols) == 0:
            # overall
            agg = {
                "unique_id": ("unique_id", "count") if "unique_id" in df.columns else ("scenario_key", "count"),
                "delta_blended": ("delta_blended", "mean"),
                "delta_EV": ("delta_EV", "sum"),
                "cross_over_to_favorite": ("cross_over_to_favorite", "sum"),
                "cross_out_to_underdog": ("cross_out_to_underdog", "sum"),
            }
            if "delta_EV_disc" in df.columns:
                agg["delta_EV_disc"] = ("delta_EV_disc", "sum")

            s = df.agg({k: v[1] for k, v in agg.items() if v[1] != "count"})
            # Count for overall
            total_n = len(df)
            row = {
                "n": total_n,
                "mean_delta_blended": float(s["delta_blended"]) if "delta_blended" in s else np.nan,
                "sum_delta_EV": float(s["delta_EV"]) if "delta_EV" in s else np.nan,
                "sum_delta_EV_disc": float(s.get("delta_EV_disc", np.nan)),
                "cross_over_to_favorite": int(df["cross_over_to_favorite"].sum()),
                "cross_out_to_underdog": int(df["cross_out_to_underdog"].sum()),
                "scenario_key": scenario_labels(scenario_key)["name"],
            }
            summaries[name] = pd.DataFrame([row])
        else:
            g = df.groupby(cols, dropna=False)
            agg_dict = {
                "n": (ID_COL, "count") if ID_COL in df.columns else ("scenario_key", "count"),
                "mean_delta_blended": ("delta_blended", "mean"),
                "sum_delta_EV": ("delta_EV", "sum"),
                "cross_over_to_favorite": ("cross_over_to_favorite", "sum"),
                "cross_out_to_underdog": ("cross_out_to_underdog", "sum"),
            }
            if "delta_EV_disc" in df.columns:
                agg_dict["sum_delta_EV_disc"] = ("delta_EV_disc", "sum")

            s = g.agg(**agg_dict).reset_index()
            s["scenario_key"] = scenario_labels(scenario_key)["name"]
            # Round floats
            for col in ["mean_delta_blended", "sum_delta_EV", "sum_delta_EV_disc"]:
                if col in s.columns:
                    s[col] = pd.to_numeric(s[col], errors="coerce").round(FLOAT_DECIMALS)
            summaries[name] = s

    return summaries

def write_reports(result_df: pd.DataFrame,
                  summaries: Dict[str, pd.DataFrame],
                  out_dir: str,
                  scenario_key: str):
    """Write CSVs and a concise Markdown report."""
    ensure_dir(out_dir)
    meta = scenario_labels(scenario_key)
    tag = meta["name"]

    # Results CSV
    results_path = os.path.join(out_dir, f"{tag}_overlay_results.csv")
    result_df.to_csv(results_path, index=False)
    info(f"Wrote results: {results_path}")

    # Grouped summaries
    for k, v in summaries.items():
        path = os.path.join(out_dir, f"{tag}_summary_{k}.csv")
        v.to_csv(path, index=False)
        info(f"Wrote summary: {path}")

    # Markdown roll-up
    md_path = os.path.join(out_dir, f"{tag}_report.md")
    total_ev_uplift = float(result_df["delta_EV"].sum())
    total_ev_disc_uplift = float(result_df["delta_EV_disc"].sum()) if "delta_EV_disc" in result_df.columns else None
    mean_delta_blended = float(result_df["delta_blended"].mean())
    crossovers = int(result_df["cross_over_to_favorite"].sum())
    crossouts = int(result_df["cross_out_to_underdog"].sum())
    n = len(result_df)

    lines = []
    lines.append(f"# {meta['pretty']} — Overlay Report")
    lines.append("")
    lines.append(f"- **Scenario key:** `{meta['name']}`")
    lines.append(f"- **Projects:** {n}")
    lines.append(f"- **Mean Δ blended probability:** {mean_delta_blended:.6f} (pp)")
    lines.append(f"- **Σ Δ EV (millions CAD):** {total_ev_uplift:.6f}")
    if total_ev_disc_uplift is not None:
        lines.append(f"- **Σ Δ EV_disc (millions CAD, PV):** {total_ev_disc_uplift:.6f}")
    lines.append(f"- **Crossovers (→ ≥0.5):** {crossovers}")
    lines.append(f"- **Cross-outs (→ <0.5):** {crossouts}")
    lines.append("")
    lines.append("## Notes")
    lines.append("- EV is computed as probability × project_cost (millions).")
    lines.append("- If present, EV_disc uses the discounted project cost column (EV_disc) as the cost base.")
    lines.append("- years_in_pipeline is standardized as `2024 - t0_orig + 1`.")
    lines.append("- MPO cap limits approvals to the configured cap and applies an odds scaling based on HAZARD_MULTIPLIER.")
    lines.append("- Indigenous overlay applies the configured time compression and hazard multiplier only to Indigenous-flagged rows.")
    lines.append("")
    with open(md_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    info(f"Wrote report: {md_path}")

def run_one(df_in: pd.DataFrame, scenario_key: str) -> pd.DataFrame:
    """
    Execute one scenario, returning the per-project result dataframe (ready to write).
    """
    meta = scenario_labels(scenario_key)

    if scenario_key == "ind_only":
        p1 = run_indigenous_overlay(df_in)
    elif scenario_key == "mpo_only":
        p1 = run_mpo_overlay(df_in)
    elif scenario_key == "combined":
        p1 = run_combined_overlay(df_in)
    else:
        raise ValueError(f"Unknown scenario '{scenario_key}'")

    res = compute_overlay_outputs(df_in, p1, scenario_key)
    return res

def main():
    global REPORT_YEAR, APPROVAL_CAP_YEARS, TIME_COMPRESSION_YEARS, HAZARD_MULTIPLIER
    # Remove argparse setup
    # parser = argparse.ArgumentParser(description="Unified Overlay Runner")
    # parser.add_argument("--input", type=str, default=INPUT_CSV, help="Path to input CSV")
    # parser.add_argument("--outdir", type=str, default=OUTPUT_DIR, help="Directory for outputs")
    # parser.add_argument("--scenario", type=str, default=SCENARIO,
    #                     choices=["ind_only", "mpo_only", "combined", "all"],
    #                     help="Which scenario to run")
    # parser.add_argument("--report_year", type=int, default=REPORT_YEAR, help="Anchor year (default 2024)")
    # parser.add_argument("--cap", type=float, default=APPROVAL_CAP_YEARS, help="MPO approvals cap (years)")
    # parser.add_argument("--time_compress", type=float, default=TIME_COMPRESSION_YEARS,
    #                     help="Indigenous overlay time compression (years; negative = faster)")
    # parser.add_argument("--hazard_mult", type=float, default=HAZARD_MULTIPLIER,
    #                     help="Hazard multiplier per year of compression (odds scaling)")
    # args = parser.parse_args()

    # Update globals from CLI (so report text matches) - now use default values or set directly
    # REPORT_YEAR = args.report_year
    # APPROVAL_CAP_YEARS = args.cap
    # TIME_COMPRESSION_YEARS = args.time_compress
    # HAZARD_MULTIPLIER = args.hazard_mult

    ensure_dir(OUTPUT_DIR)

    # Load input
    if not os.path.exists(INPUT_CSV):
        raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV}")
    df = pd.read_csv(INPUT_CSV)

    # Standard fields
    df = attach_standard_fields(df)

    # Sanity checks
    missing_crit = [c for c in [PROB_COL, COST_COL] if c not in df.columns]
    if missing_crit:
        raise ValueError(f"Missing required column(s): {missing_crit}. "
                         f"Expected at least probability '{PROB_COL}' and cost '{COST_COL}'.")

    # Run scenarios
    scenarios_to_run = ["ind_only", "mpo_only", "combined"] if SCENARIO == "all" else [SCENARIO]

    for scen in scenarios_to_run:
        info(f"Running scenario: {scen}")
        res = run_one(df, scen)
        sums = group_summaries(res, scen)
        write_reports(res, sums, OUTPUT_DIR, scen)

    info("Done.")

if __name__ == "__main__":
    main()

[WARN] 't0_orig' missing; cannot compute years_in_pipeline. Will default to NaN.
[WARN] Grouping column 'FOAK' missing; filling with NA.
[WARN] Grouping column 'cost_band' missing; filling with NA.
[WARN] No Indigenous flag column found; ind_only overlay will behave like no-op.


[INFO] Running scenario: ind_only
[INFO] Wrote results: overlay_outputs/ind_only_overlay_results.csv
[INFO] Wrote summary: overlay_outputs/ind_only_summary_province.csv
[INFO] Wrote summary: overlay_outputs/ind_only_summary_sector.csv
[INFO] Wrote summary: overlay_outputs/ind_only_summary_group.csv
[INFO] Wrote summary: overlay_outputs/ind_only_summary_foak.csv
[INFO] Wrote summary: overlay_outputs/ind_only_summary_cleantech.csv
[INFO] Wrote summary: overlay_outputs/ind_only_summary_cost_band.csv
[INFO] Wrote summary: overlay_outputs/ind_only_summary_overall.csv
[INFO] Wrote report: overlay_outputs/ind_only_report.md
[INFO] Running scenario: mpo_only
[INFO] Wrote results: overlay_outputs/mpo_only_overlay_results.csv
[INFO] Wrote summary: overlay_outputs/mpo_only_summary_province.csv
[INFO] Wrote summary: overlay_outputs/mpo_only_summary_sector.csv
[INFO] Wrote summary: overlay_outputs/mpo_only_summary_group.csv
[INFO] Wrote summary: overlay_outputs/mpo_only_summary_foak.csv
[INFO] Wrot

[WARN] No Indigenous flag column found; ind_only overlay will behave like no-op.


In [ ]:
import pandas as pd

# Load files
df_step_a_result = pd.read_csv('/content/overlay_outputs/mpi_ind_overlay_results.csv')
df_ev_target = pd.read_csv('/content/mpi_2024_ev_dev_combined_5y.csv')

# --- CORRECTED MERGE ---
# Use 'Unique ID' (capitalized) for both sides
merged_df = df_ev_target.merge(
    df_step_a_result[['Unique ID', 'blended_overlay', 'EV_overlay']],
    on='Unique ID',  # Safer: joins on the exact same column name
    how='left'
)

# Overwrite the probability column
merged_df['blended_prob'] = merged_df['blended_overlay']

# Clean up columns (remove the temporary merge columns)
final_input_df = merged_df.drop(columns=['blended_overlay', 'EV_overlay'])

# Save for Step B
final_input_df.to_csv('mpi_step_b_input_ready.csv', index=False)
print("SUCCESS: Corrected file 'mpi_step_b_input_ready.csv' created.")

SUCCESS: Corrected file 'mpi_step_b_input_ready.csv' created.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Government Override + Contracted-Demand Runner — v2 (Discounted EV, Evidence Tags, Sensitivity Bands)

Scenario-only overlays applied to an already-scored MPI cohort (NO re-estimation of Bayes/Cox).

What’s new vs v1:
- EV basis: uses EV_disc if present (discounted EV); falls back to EV_cum only if EV_disc missing.
- Anchors exposed as stylized *scenario dials* (conservative/central/optimistic) with optional sensitivity table.
- Interaction bump: parameterized via theta_scale (small, explicit), reported in summary.
- Evidence tags + disclaimer embedded in outputs; audit JSON written.
- Performance hygiene (batch concat) and QA metrics (cap hits, crossers).
"""

import os, math, argparse, json
import numpy as np
import pandas as pd

# ==================== CONFIG (defaults; override via CLI) ====================
CONFIG = {
    # Inputs (CSV must include: p_bayes, p_cox, blended_prob; EV_disc preferred, EV_cum acceptable)
    "in_csv": "mpi_step_b_input_ready.csv",

    # Outputs
    "out_dir": "./overlay_out",

    # Override strength grid (α) & Contract strength grid (β)
    "alphas": [0.25, 0.50, 0.75, 1.00],                  # Guidance → Emergency
    "betas":  [0.00, 0.25, 0.50, 0.75, 1.00],            # Off → Max

    # Interaction knob between override & contracts (−1=substitution, 0=independent, +1=complementarity)
    "theta": 0.00,
    # Small scalar for logit-bump magnitude (explicit, reported)
    "theta_scale": 0.05,

    # Probability cap & crosser threshold
    "cap_p": 0.90,
    "threshold": 0.50,

    # Anchors (stylized scenario dials). v2 uses the "central" set for main run, but records bands.
    "anchors_bands": {
        "conservative": {"OR_override": math.exp(0.45), "HR_override": math.exp(0.30), "k_contract": 0.50, "h_contract": 0.35},
        "central":      {"OR_override": math.exp(0.75), "HR_override": math.exp(0.50), "k_contract": 0.75, "h_contract": 0.50},
        "optimistic":   {"OR_override": math.exp(1.00), "HR_override": math.exp(0.75), "k_contract": 1.00, "h_contract": 0.65},
    },
    "anchors_use": "central",  # which band to use for the main run

    # Evidence tags & disclaimer
    "evidence_timeline":   "moderate",
    "evidence_contracts":  "weak",
    "evidence_indigenous": "weak",
    "disclaimer": ("Counterfactual overlays; no re-estimation of Bayes/Cox; "
                   "stylized scenario dials with sensitivity bands; not a forecast."),

    # Sensitivity table toggle (writes a compact CSV comparing bands at α=0.5, β=0.5)
    "write_anchor_sensitivity": True,

    # Plot toggle (not producing charts here; keep code minimal)
}

# ==================== Labels ====================
LABELS        = {0.25: "Guidance", 0.50: "Priority", 0.75: "Mandate", 1.00: "Emergency"}
BETA_LABELS   = {0.00: "Off", 0.25: "Light", 0.50: "Medium", 0.75: "Strong", 1.00: "Max"}

# ==================== Helpers ====================
def clamp01(x):
    try:
        v = float(x)
        if not np.isfinite(v): return 0.0
        return float(np.clip(v, 0.0, 1.0))
    except Exception:
        return 0.0

def safe_prob(p):
    return float(np.clip(float(p), 1e-9, 1.0 - 1e-9))

def write_csv(df: pd.DataFrame, path: str):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    df.to_csv(path, index=False)

def estimate_w_bayes_row(p_bayes, p_cox, p_blend):
    """Estimate per-row Bayes blend weight from the three legs (no refit)."""
    pb = safe_prob(p_bayes); pc = safe_prob(p_cox)
    denom = pb - pc
    if abs(denom) < 1e-12:
        return 0.60
    w = (p_blend - pc) / denom
    return float(np.clip(w, 0.0, 1.0))

# ==================== Transforms ====================
def apply_override_scalar(p_bayes, p_cox, w_bayes, alpha, OR_anchor, HR_anchor, cap):
    """Post-model override lever with strength α: Bayes logit shift & Cox survival power; then re-blend."""
    pb = safe_prob(p_bayes); pc = safe_prob(p_cox); w = float(np.clip(w_bayes, 0.0, 1.0))
    # Bayes
    OR_eff   = OR_anchor ** alpha
    logit_pb = math.log(pb / (1 - pb))
    pb_adj   = 1 / (1 + math.exp(-(logit_pb + math.log(OR_eff))))
    # Cox
    HR_eff = HR_anchor ** alpha
    S      = 1 - pc
    S_adj  = S ** HR_eff
    pc_adj = 1 - S_adj
    # Blend & cap
    p_blend = float(np.clip(w * pb_adj + (1 - w) * pc_adj, 0.0, cap))
    return pb_adj, pc_adj, p_blend

def apply_contract_overlay_row(row, beta, k, h, cap):
    """Contracted-demand overlay scaled by coverage in [0,1], intensity β."""
    coverage = clamp01(row.get("contract_coverage_share", 0.0))
    if beta <= 0.0 or coverage <= 1e-12:
        return row["p_bayes"], row["p_cox"], row["blended_prob"]

    pb = safe_prob(row["p_bayes"]); pc = safe_prob(row["p_cox"]); w = float(np.clip(row["w_bayes_est"], 0.0, 1.0))

    OR_c = math.exp(k * coverage) ** beta
    HR_c = math.exp(h * coverage) ** beta

    logit_pb = math.log(pb / (1 - pb))
    pb_adj   = 1 / (1 + math.exp(-(logit_pb + math.log(OR_c))))

    S      = 1 - pc
    S_adj  = S ** HR_c
    pc_adj = 1 - S_adj

    p_blend = float(np.clip(w * pb_adj + (1 - w) * pc_adj, 0.0, cap))
    return pb_adj, pc_adj, p_blend

def _slogit(p):
    p = safe_prob(p)
    return math.log(p / (1 - p))

def compose_with_interaction(p_base, p_override, p_contract, theta, theta_scale, cap):
    """Compose two levers: sequential probability + small logit-bump scaled by theta_scale."""
    p_base = safe_prob(p_base); p_over = safe_prob(p_override); p_cont = safe_prob(p_contract)
    # Sequential baseline (bounded)
    p_seq = float(np.clip(p_over + p_cont - p_base, 0.0, cap))
    if abs(theta) < 1e-12 or abs(theta_scale) < 1e-12:
        return p_seq
    # Small bump in logit space, scaled
    logit_base = _slogit(p_base)
    bump = theta_scale * (_slogit(p_over) - logit_base) * (_slogit(p_cont) - logit_base)
    logit_final = _slogit(p_seq) + theta * bump
    p_final = 1 / (1 + math.exp(-logit_final))
    return float(np.clip(p_final, 0.0, cap))

# ==================== Core ====================
def run_override_contract(in_csv, out_dir, alphas, betas, theta, theta_scale, cap_p, threshold, anchors_use, anchors_bands, write_anchor_sensitivity):
    df = pd.read_csv(in_csv)

    # Required columns
    for col in ["p_bayes", "p_cox", "blended_prob"]:
        if col not in df.columns:
            raise ValueError(f"Input must include '{col}' column. Missing in file: {in_csv}")

    # EV column preference
    ev_col = "EV_disc" if "EV_disc" in df.columns else ("EV_cum" if "EV_cum" in df.columns else None)
    if ev_col is None:
        raise ValueError("Input must include EV_disc (preferred) or EV_cum.")

    # Copy & prep
    out = df.copy()

    # Per-row blend weights (no model refit)
    out["w_bayes_est"] = out.apply(lambda r: estimate_w_bayes_row(r["p_bayes"], r["p_cox"], r["blended_prob"]), axis=1).fillna(0.60)

    # Normalize contract inputs
    out["ppa_flag"]         = out.get("ppa_flag", 0)
    out["gov_ofstake_flag"] = out.get("gov_ofstake_flag", 0)

    # Ensure clamp01 is applied correctly to merchant_share and contract_coverage_share
    merchant_share_data = out.get("merchant_share", 0.0)
    if isinstance(merchant_share_data, pd.Series):
        out["merchant_share"] = merchant_share_data.apply(clamp01)
    else:
        out["merchant_share"] = clamp01(merchant_share_data)

    cov = out.get("contract_coverage_share", np.nan)
    if isinstance(cov, pd.Series):
        out["contract_coverage_share"] = cov.where(pd.notna(cov), 1.0 - out["merchant_share"]).apply(clamp01)
    else:
        out["contract_coverage_share"] = clamp01(cov if pd.notna(cov) else 1.0 - out["merchant_share"])


    # Anchors (select band)
    band = anchors_bands.get(anchors_use, anchors_bands["central"])
    OR_OVERRIDE_ANCHOR = float(band["OR_override"])
    HR_OVERRIDE_ANCHOR = float(band["HR_override"])
    K_CONTRACT         = float(band["k_contract"])
    H_CONTRACT         = float(band["h_contract"])

    # ---------- Override-only (α) ----------
    # Batch-compute columns into a dict, then concat (performance hygiene)
    over_cols = {}
    pv_success = np.where(out["blended_prob"] > 1e-12, out[ev_col] / out["blended_prob"], np.nan)

    for a in alphas:
        res = out.apply(
            lambda r: apply_override_scalar(r["p_bayes"], r["p_cox"], r["w_bayes_est"], a,
                                            OR_OVERRIDE_ANCHOR, HR_OVERRIDE_ANCHOR, cap_p),
            axis=1, result_type="expand"
        )
        p_ba, p_ca, p_bl = res[0].values, res[1].values, res[2].values
        over_cols[f"p_bayes_{a}"]  = p_ba
        over_cols[f"p_cox_{a}"]    = p_ca
        over_cols[f"blended_{a}"]  = p_bl
        dp = p_bl - out["blended_prob"].values
        over_cols[f"Δp_{a}"]       = dp
        EVa = pv_success * p_bl
        over_cols[f"{ev_col}_{a}"] = EVa
        over_cols[f"ΔEV_{a}"]      = EVa - out[ev_col].values

    out = pd.concat([out, pd.DataFrame(over_cols, index=out.index)], axis=1)

    # ---------- Contract-only (β) ----------
    cont_cols = {}
    for b in betas:
        if b == 0.0:
            cont_cols[f"p_bayes_contract_{b}"]   = out["p_bayes"].values
            cont_cols[f"p_cox_contract_{b}"]     = out["p_cox"].values
            cont_cols[f"blended_contract_{b}"]    = out["blended_prob"].values
        else:
            cres = out.apply(lambda r: apply_contract_overlay_row(r, b, K_CONTRACT, H_CONTRACT, cap_p),
                             axis=1, result_type="expand")
            cont_cols[f"p_bayes_contract_{b}"] = cres[0].values
            cont_cols[f"p_cox_contract_{b}"]   = cres[1].values
            cont_cols[f"blended_contract_{b}"] = cres[2].values

    out = pd.concat([out, pd.DataFrame(cont_cols, index=out.index)], axis=1)

    # ---------- Compose α × β with interaction θ ----------
    comp_cols = {}
    cap_hits = 0

    for a in alphas:
        p_over = out[f"blended_{a}"].values
        for b in betas:
            p_cont = out[f"blended_contract_{b}"].values
            blended = np.fromiter(
                (compose_with_interaction(pb, po, pc, theta, theta_scale, cap_p)
                 for pb, po, pc in zip(out["blended_prob"].values, p_over, p_cont)),
                dtype=float, count=len(out)
            )
            comp_cols[f"blended_alpha_{a}_beta_{b}"] = blended
            dp = blended - out["blended_prob"].values
            comp_cols[f"Δp_alpha_{a}_beta_{b}"] = dp
            EVab = pv_success * blended
            comp_cols[f"{ev_col}_alpha_{a}_beta_{b}"] = EVab
            comp_cols[f"ΔEV_alpha_{a}_beta_{b}"]      = EVab - out[ev_col].values
            cap_hits += int((blended >= (cap_p - 1e-9)).sum())

    out = pd.concat([out, pd.DataFrame(comp_cols, index=out.index)], axis=1)

    # ---------- Write per-project ----------
    os.makedirs(out_dir, exist_ok=True)
    per_project_csv = os.path.join(out_dir, "override_contract_overlay_full.csv")
    write_csv(out, per_project_csv)

    # ---------- Summaries ----------
    base_EV_sum   = float(np.nansum(out[ev_col].values))
    base_crossers = int((out["blended_prob"] >= threshold).sum())

    # Override-only summary (by α)
    rows = []
    for a in alphas:
        dp = out[f"Δp_{a}"]
        dE = out[f"ΔEV_{a}"]
        rows.append({
            "alpha": a,
            "alpha_label": LABELS.get(a, str(a)),
            "Δp_mean": float(np.nanmean(dp.values)),
            "Δp_median": float(np.nanmedian(dp.values)),
            "Δp_p10": float(np.nanpercentile(dp.values, 10)),
            "Δp_p90": float(np.nanpercentile(dp.values, 90)),
            "ΔEV_sum": float(np.nansum(dE.values)),
            "uplift_%_EV": (float(np.nansum(dE.values)) / base_EV_sum) if base_EV_sum else np.nan,
            "net_new_crossers": int((out[f"blended_{a}"] >= threshold).sum() - base_crossers)
        })
    over_sum = pd.DataFrame(rows)
    write_csv(over_sum, os.path.join(out_dir, "override_summary_by_alpha.csv"))

    # Grid summary (α × β, with θ)
    rows = []
    for a in alphas:
        for b in betas:
            dp = out[f"Δp_alpha_{a}_beta_{b}"]
            dE = out[f"ΔEV_alpha_{a}_beta_{b}"]
            rows.append({
                "alpha": a, "alpha_label": LABELS.get(a, str(a)),
                "beta": b,  "beta_label":  BETA_LABELS.get(b, str(b)),
                "THETA": float(theta), "theta_scale": float(theta_scale),
                "Δp_mean": float(np.nanmean(dp.values)),
                "Δp_median": float(np.nanmedian(dp.values)),
                "Δp_p10": float(np.nanpercentile(dp.values, 10)),
                "Δp_p90": float(np.nanpercentile(dp.values, 90)),
                "ΔEV_sum": float(np.nansum(dE.values)),
                "uplift_%_EV": (float(np.nansum(dE.values)) / base_EV_sum) if base_EV_sum else np.nan,
                "net_new_crossers": int((out[f"blended_alpha_{a}_beta_{b}"] >= threshold).sum() - base_crossers),
            })
    grid_sum = pd.DataFrame(rows)
    write_csv(grid_sum, os.path.join(out_dir, "override_contract_grid_summary.csv"))

    # ---------- Optional sensitivity table over anchor bands (α=0.5, β=0.5) ----------
    anchor_sens_path = None
    if write_anchor_sensitivity:
        a_sens, b_sens = 0.50, 0.50
        sens_rows = []
        for band_name, anchors in anchors_bands.items():
            ORa = anchors["OR_override"]; HRa = anchors["HR_override"]
            ka  = anchors["k_contract"];   ha  = anchors["h_contract"]

            # One-shot recompute for the pair (α, β) using alternative anchors
            p_over_tmp = np.fromiter(
                (apply_override_scalar(r["p_bayes"], r["p_cox"], r["w_bayes_est"],
                                       a_sens, ORa, HRa, cap_p)[2]
                 for _, r in out.iterrows()), dtype=float, count=len(out)
            )
            p_cont_tmp = np.fromiter(
                (apply_contract_overlay_row(r, b_sens, ka, ha, cap_p)[2]
                 for _, r in out.iterrows()), dtype=float, count=len(out)
            )
            blend_tmp  = np.fromiter(
                (compose_with_interaction(pb, po, pc, theta, theta_scale, cap_p)
                 for pb, po, pc in zip(out["blended_prob"].values, p_over_tmp, p_cont_tmp)),
                dtype=float, count=len(out)
            )
            dp_tmp = blend_tmp - out["blended_prob"].values
            EV_tmp = pv_success * blend_tmp
            dE_tmp = EV_tmp - out[ev_col].values
            sens_rows.append({
                "band": band_name,
                "OR_override": ORa, "HR_override": HRa, "k_contract": ka, "h_contract": ha,
                "alpha": a_sens, "beta": b_sens,
                "Δp_mean": float(np.nanmean(dp_tmp)), "Δp_median": float(np.nanmedian(dp_tmp)),
                "ΔEV_sum": float(np.nansum(dE_tmp)),
                "uplift_%_EV": (float(np.nansum(dE_tmp)) / base_EV_sum) if base_EV_sum else np.nan
            })
        anchor_sens = pd.DataFrame(sens_rows)
        anchor_sens_path = os.path.join(out_dir, "override_anchor_sensitivity.csv")
        write_csv(anchor_sens, anchor_sens_path)

    # ---------- Audit JSON ----------
    audit = {
        "rows": int(len(out)),
        "ev_basis": f"{ev_col} (discounted basis preferred)" if ev_col == "EV_disc" else "EV_cum (fallback)",
        "cap_p": float(cap_p),
        "cap_hits": int(cap_hits),
        "threshold": float(threshold),
        "alphas": list(map(float, alphas)),
        "betas":  list(map(float, betas)),
        "theta": float(theta),
        "theta_scale": float(theta_scale),
        "anchors_used": anchors_use,
        "anchors_bands": anchors_bands,
        "evidence": {
            "timeline":   CONFIG["evidence_timeline"],
            "contracts":  CONFIG["evidence_contracts"],
            "indigenous": CONFIG["evidence_indigenous"]
        },
        "disclaimer": CONFIG["disclaimer"],
        "outputs": {
            "per_project_csv": per_project_csv,
            "override_summary_by_alpha": os.path.join(out_dir, "override_summary_by_alpha.csv"),
            "override_contract_grid_summary": os.path.join(out_dir, "override_contract_grid_summary.csv"),
            "anchor_sensitivity_csv": anchor_sens_path
        }
    }
    with open(os.path.join(out_dir, "override_run_audit.json"), "w", encoding="utf-8") as f:
        json.dump(audit, f, indent=2)

    # Console peek
    print("\n=== Override summary by α ===")
    print(over_sum.to_string(index=False))
    print("\n=== Grid summary (α × β) — head ===")
    print(grid_sum.head(12).to_string(index=False))
    print(f"\nPer-project results: {per_project_csv}")
    print(f"Summaries in: {out_dir}")
    if anchor_sens_path:
        print(f"Anchor sensitivity table: {anchor_sens_path}")

# ==================== CLI ====================
# Remove argparse setup
# def parse_args():
#     p = argparse.ArgumentParser(description="Government Override + Contracted Demand Runner — v2")
#     p.add_argument("--in_csv", help="Input cohort CSV")
#     p.add_argument("--out_dir", help="Output directory")
#     p.add_argument("--alphas", help="Comma-separated α list, e.g., 0.25,0.5,0.75,1.0")
#     p.add_argument("--betas", help="Comma-separated β list, e.g., 0.0,0.25,0.5,0.75,1.0")
#     p.add_argument("--theta", type=float, help="Interaction knob (−1..+1)")
#     p.add_argument("--theta_scale", type=float, help="Small scalar for logit bump")
#     p.add_argument("--cap_p", type=float, help="Probability cap (default 0.90)")
#     p.add_argument("--threshold", type=float, help="Crosser threshold (default 0.50)")
#     p.add_argument("--anchors_use", choices=["conservative","central","optimistic"], help="Band for anchors")
#     p.add_argument("--write_anchor_sensitivity", type=int, choices=[0,1], help="Write anchor sensitivity CSV (1/0)")
#     return p.parse_args()

# def _parse_list(val):
#     return [float(x) for x in str(val).split(",") if str(x).strip()]

if __name__ == "__main__":
    # args = parse_args() # Removed argparse
    # overlay CLI args into CONFIG # Removed argparse
    # if args.in_csv:     CONFIG["in_csv"] = args.in_csv
    # if args.out_dir:    CONFIG["out_dir"] = args.out_dir
    # if args.alphas:     CONFIG["alphas"] = _parse_list(args.alphas)
    # if args.betas:      CONFIG["betas"]  = _parse_list(args.betas)
    # if args.theta is not None:        CONFIG["theta"] = float(args.theta)
    # if args.theta_scale is not None:  CONFIG["theta_scale"] = float(args.theta_scale)
    # if args.cap_p is not None:        CONFIG["cap_p"] = float(args.cap_p)
    # if args.threshold is not None:    CONFIG["threshold"] = float(args.threshold)
    # if args.anchors_use:              CONFIG["anchors_use"] = args.anchors_use
    # if args.write_anchor_sensitivity is not None:
    #     CONFIG["write_anchor_sensitivity"] = bool(args.write_anchor_sensitivity)

    # Use default values from CONFIG directly or set explicitly here
    run_override_contract(
        in_csv=CONFIG["in_csv"],
        out_dir=CONFIG["out_dir"],
        alphas=CONFIG["alphas"],
        betas=CONFIG["betas"],
        theta=CONFIG["theta"],
        theta_scale=CONFIG["theta_scale"],
        cap_p=CONFIG["cap_p"],
        threshold=CONFIG["threshold"],
        anchors_use=CONFIG["anchors_use"],
        anchors_bands=CONFIG["anchors_bands"],
        write_anchor_sensitivity=CONFIG["write_anchor_sensitivity"],
    )


=== Override summary by α ===
 alpha alpha_label  Δp_mean  Δp_median    Δp_p10   Δp_p90      ΔEV_sum  uplift_%_EV  net_new_crossers
  0.25    Guidance 0.017733   0.019468 -0.000912 0.034977  5513.094027     0.044724                13
  0.50    Priority 0.038177   0.038924  0.003156 0.069645 11661.632801     0.094602                30
  0.75     Mandate 0.059081   0.059283  0.012185 0.103969 17885.292115     0.145090                41
  1.00   Emergency 0.080455   0.080353  0.018414 0.137563 24219.783262     0.196477                51

=== Grid summary (α × β) — head ===
 alpha alpha_label  beta beta_label  THETA  theta_scale  Δp_mean  Δp_median    Δp_p10   Δp_p90      ΔEV_sum  uplift_%_EV  net_new_crossers
  0.25    Guidance  0.00        Off    0.0         0.05 0.017733   0.019468 -0.000912 0.034977  5513.094027     0.044724                13
  0.25    Guidance  0.25      Light    0.0         0.05 0.017733   0.019468 -0.000912 0.034977  5513.094027     0.044724                13
  0.2